# Model Serving — API client exploration

The service lives in `../src/main.py`. Start it first (from the subtopic folder):
```
uvicorn src.main:app --reload          # or:
docker build -t serving-demo . && docker run -p 8000:8000 serving-demo
```
Then run this notebook to exercise it like a client would.

## 1. Health check

In [ ]:
import httpx

BASE = "http://localhost:8000"
resp = httpx.get(f"{BASE}/health", timeout=5)
print(resp.status_code, resp.json())

## 2. Valid predictions

In [ ]:
for features in [[1.0, 2.0, 3.0], [0.5, 0.5, 0.5], [10, -3, 2.2]]:
    r = httpx.post(f"{BASE}/predict", json={"features": features})
    print(features, "->", r.status_code, r.json())

## 3. Invalid input: Pydantic rejects it before your handler runs (expect 422)

In [ ]:
bad_payloads = [
    {},                                   # missing field
    {"features": "not-a-list"},           # wrong type
    {"features": [1, "two", 3]},          # wrong element type
]
for payload in bad_payloads:
    r = httpx.post(f"{BASE}/predict", json=payload)
    print(r.status_code, r.json()["detail"][0]["msg"] if r.status_code == 422 else r.json())

## 4. Basic latency check: p50 / p95 / p99 over N requests

In [ ]:
import time

import numpy as np

latencies_ms = []
with httpx.Client() as client:  # reuse one connection, like a real caller
    for _ in range(200):
        t0 = time.perf_counter()
        client.post(f"{BASE}/predict", json={"features": [1.0, 2.0, 3.0]})
        latencies_ms.append((time.perf_counter() - t0) * 1000)

for q in [50, 95, 99]:
    print(f"p{q}: {np.percentile(latencies_ms, q):.2f} ms")

## 5. Mini-project notes

Record your observations here after swapping the toy `sum()` in `src/main.py`
for a real model (e.g. loaded from the MLflow registry):

- p50 / p95 / p99 with the real model: …
- Effect of `docker run` vs. bare uvicorn: …
- What happens under concurrent load (try `locust` or many threads): …